### Dataset creation

In [ ]:
from openai import OpenAI
import datasets
import os
import csv
import ast
import random

In [ ]:
def format_input(input):
    input = [f"[{i}] {message}" for i, message in enumerate(input, 1)]
    input = "\n".join(input)
    return input

def extract_scores(output):
    message = output.choices[0].message.content
    scores = ast.literal_eval(message)
    assert len(scores) == 10
    return scores

def retrieve_scores(input):
    system = (
        "You are an expert data annotator. Your task is to evaluate the NEGATIVE EMOTIONAL INTENSITY of exactly 10 user messages.\n\n"
        "SCORING SCALE:\n"
        "0.0 = No discernible negative emotion (neutral, positive, or purely informational)\n"
        "0.2 = Slight annoyance, mild concern, or impatience\n"
        "0.4 = Noticeable dissatisfaction, disappointment, or irritation\n"
        "0.6 = Strong frustration, upset, or anger\n"
        "0.8 = Very strong anger, distress, or exasperation\n"
        "1.0 = Extreme negative emotional expression (rage, panic, severe distress)\n\n"
        "STRICT RULES:\n"
        "1. You will receive EXACTLY 10 messages. You must return EXACTLY 10 scores.\n"
        "2. Maintain the exact 1-to-1 order of the input messages.\n"
        "3. Score the explicit emotion in the text, not the implied situation.\n"
        "4. A simple request for help, a greeting, or a neutral statement is 0.0.\n"
        "5. Ignore positive emotions unless negative emotion is also present.\n\n"
        "OUTPUT FORMAT:\n"
        "Return ONLY a raw Python list of exactly 10 floats.\n"
        "Do NOT include markdown formatting, explanations, or any other text.\n"
        "Example: [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 0.0, 0.2, 0.4, 0.0]"
    )
    input = format_input(input)
    
    output = client.chat.completions.create(
        model="deepseek-flash",
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": input},
        ],
        stream=False,
        reasoning_effort="low",
        extra_body={"thinking": {"type": "enabled"}}
    )
    scores = extract_scores(output)
    
    return scores

In [ ]:
# Build an object with which we can call our llm of choice

client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
)

In [ ]:
# Load a dataset and extract the text from both the training and the testing set into one big list.
# Shuffle the list just to make sure that there's no correlation between sequential items.

dataset = datasets.load_dataset("mteb/toxic_conversations_50k")
input = list(dataset["train"]["text"]) + list(dataset["test"]["text"])
input = [message for message in input if len(message) < 500]

random.seed(67)
random.shuffle(input)

In [ ]:
# Build the dataset by fetching a score for each message by prompting an llm in batches of 10 items.
# Join the score and the message and write each row into dataset.csv.

amount = 10000

with open("dataset.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    for i in range(0, amount, 10):
        print(f"{i}/{amount}")
        j = i + 10
        sample = input[i:j]
        scores = retrieve_scores(sample)
        rows = zip(scores,sample)
        writer.writerows(rows)

In [ ]:
# Pick a sample of 10 items from the dataset and print it 

with open("dataset.csv", "r", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    rows = list(reader)
    random.seed(67)
    sample = random.sample(rows, 10)
    sample = "\n".join(f"[{score}] {message.replace("\n", "")}" for score, message in sample)
    
    print(sample)

In [ ]:
# Print statistical information about the dataset

with open("dataset.csv", "r", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    rows = list(reader)
    rows_amount = len(rows)
    scores = [float(score) for score, _ in rows]
    mean = sum(scores) / rows_amount
    median = scores[rows_amount // 2]
    std = (sum((score - mean) ** 2 for score in scores) / rows_amount) ** 0.5

    print(f"ROWS:   {rows_amount}")
    print(f"MEDIAN: {median:.4}")
    print(f"MEAN:   {mean:.4}")
    print(f"STD:    {std:.4}")

### Transformer training

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, DataCollatorWithPadding
import datasets

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.squeeze()
    mse = ((labels - predictions) ** 2).sum() / predictions.shape[0]
    
    return {"MSE": mse}

In [ ]:
# Build the tokenizer and the model, make sure that the model is using cuda if available

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-cased")
model = AutoModelForSequenceClassification.from_pretrained(
    "google-bert/bert-base-cased",
    device_map="auto",
    num_labels=1,
    problem_type="regression",
)

device = next(model.parameters()).device.type
print(f"DEVICE: {device}")

In [ ]:
# Load the dataset from the file saved earlier, split it into a training and a testing set and tokenize each.

dataset = datasets.load_dataset(
    "csv",
    data_files="dataset.csv",
    column_names=["labels", "text"]
)
dataset = dataset["train"].train_test_split(0.2, seed=67)
dataset = dataset.map(tokenize, batched=True)

In [ ]:
# Build the trainer. You might only need to change bf16 and the batch size depending on your hardware.

args = TrainingArguments(
    bf16=True,
    eval_strategy="epoch",
    learning_rate=5e-5,
    num_train_epochs=3,
    optim="adamw_torch_fused",
    per_device_eval_batch_size=16,
    per_device_train_batch_size=16,
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
)
 
trainer = Trainer(
    args=args,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    model=model,
    eval_dataset=dataset["test"],
    train_dataset=dataset["train"],
)

In [ ]:
# Train and save the trained model

trainer.train()
trainer.save_model("bert-finetuned")

### Inference

In [ ]:
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer

In [ ]:
# Load the tokenizer and the model from the saved file. Build a classifier pipeline utilizing them.

tokenizer = AutoTokenizer.from_pretrained("bert-finetuned")
model = AutoModelForSequenceClassification.from_pretrained("bert-finetuned")
classifier = pipeline(
    task="text-classification",
    model=model,
    tokenizer=tokenizer,
)

In [ ]:
# Test the model using two prompts, the latter of which is intended to be emotionally charged unlike the former one.

a = classifier("forbidden snack is edible")[0]["score"]
b = classifier("forbidden snack is absolutely fucking disgusting")[0]["score"]

print(a, b)